# 07. Rating Packet Export

Local-only entry point for the six-document R1 development pilot. Uses the shared Python builder; no Gemini requests or LangSmith traces are made.

Expected packets: 18 assessment-quality, 18 feedback-quality, and 6 feedback-implied score packets. Human feedback comes from the config-pinned derived CSV; this notebook does not read the Excel workbook directly or verify that conversion.

Run from the `prebi_v1` root with its `.venv` kernel. Only counts are printed during preflight. Packet text and the condition key remain in memory or protected storage, never notebook outputs. Protocols are development drafts, not frozen confirmatory instruments.

In [4]:
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv
from reflection_assessment_feedback.rating_packets import build_development_rating_packet_bundle

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "prebi_v1":
    raise RuntimeError("Run this notebook from the prebi_v1 project root.")
load_dotenv(PROJECT_ROOT / ".env", override=False)
root_setting = os.environ.get("PREBI_DATA_ROOT")
if not root_setting:
    raise RuntimeError("Set PREBI_DATA_ROOT in the local .env or environment.")
PROTECTED_ROOT = Path(root_setting).expanduser().resolve()
WRITE_PACKETS = True
OUTPUT_DIRECTORY = PROTECTED_ROOT / "rating-packets" / "development"
PACKET_PATH = OUTPUT_DIRECTORY / "r1-rating-packets.json"
KEY_PATH = OUTPUT_DIRECTORY / "r1-condition-key.json"
print(f"Local setup ready. Packet export enabled: {WRITE_PACKETS}.")

Local setup ready. Packet export enabled: True.


In [5]:
packet_bundle, condition_key = build_development_rating_packet_bundle(PROTECTED_ROOT, write=False)
expected_counts = {"assessment_quality": 18, "feedback_quality": 18, "feedback_implied_score": 6}
task_counts = Counter(packet["task"] for packet in packet_bundle["packets"])
if task_counts != expected_counts:
    raise ValueError("Packet task counts differ from the expected R1 pilot.")
packet_ids = {packet["packet_id"] for packet in packet_bundle["packets"]}
if len(packet_ids) != 42 or len(condition_key["records"]) != 42:
    raise ValueError("Packet or key identifiers are missing or duplicated.")
if packet_ids != {record["packet_id"] for record in condition_key["records"]}:
    raise ValueError("Packet bundle and protected key do not match.")
for packet in packet_bundle["packets"]:
    if packet["condition_blinded"] is not True:
        raise ValueError("An unblinded packet was returned.")
    if {"run_id", "document_id", "condition", "model_name", "analysis", "gold"}.intersection(packet):
        raise ValueError("A packet contains protected provenance fields.")
print("Local preflight passed. No files written.")
print(dict(task_counts))

Local preflight passed. No files written.
{'assessment_quality': 18, 'feedback_quality': 18, 'feedback_implied_score': 6}


## Export and verify

To write the approved local pilot, set `WRITE_PACKETS = True` in Cell 2, then rerun Cells 2, 3, and 5. The builder rebuilds fresh packet IDs when exporting and refuses to overwrite either existing export file. Preflight packet IDs are temporary and must not be used for assignments.

Cell 5 reads the exported files back and compares them with the written snapshots. Upload only `r1-rating-packets.json` into the standalone rating UI. Keep `r1-condition-key.json` separately restricted and never distribute it to raters. Both files contain protected research data; do not place them in the repository or frontend build. Preserve the exports once annotation begins.

In [6]:
if not WRITE_PACKETS:
    print("Export disabled. No files written. Set WRITE_PACKETS=True in Cell 2 to export deliberately.")
else:
    packet_bundle, condition_key = build_development_rating_packet_bundle(PROTECTED_ROOT, write=True)
    saved_bundle = json.loads(PACKET_PATH.read_text(encoding="utf-8"))
    saved_key = json.loads(KEY_PATH.read_text(encoding="utf-8"))
    if saved_bundle != packet_bundle or saved_key != condition_key:
        raise ValueError("Saved exports differ from the written snapshots.")
    print("Export completed and read-back verified.")
    print(dict(Counter(packet["task"] for packet in saved_bundle["packets"])))
    print(f"Upload only this packet bundle: {PACKET_PATH}")
    print("Condition key saved separately in protected storage. Never upload it to the rating UI.")

Export completed and read-back verified.
{'assessment_quality': 18, 'feedback_quality': 18, 'feedback_implied_score': 6}
Upload only this packet bundle: /Users/quynguyen/Documents/CodingProjects/prebi_data/rating-packets/development/r1-rating-packets.json
Condition key saved separately in protected storage. Never upload it to the rating UI.
